In [ ]:
import os

path = '/share/home/dq076/data/cases/globe/g1/history/g1_hist_1951.nc'
new_path = '/share/home/dq076/data/cases/globe/g1/postdata/'
os.makedirs(new_path, exist_ok=True)
os.system(f'cdo -selname,f_annavg_somhr {path} {new_path}annavg_somhr.nc')

In [ ]:
%matplotlib inline

import os
import cmaps
import numpy as np
import xarray as xr
import pandas as pd
import netCDF4 as nc
import rioxarray as rxr
import geopandas as gpd
from pylab import rcParams
# import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.patches as patches
import matplotlib.font_manager as fm
from matplotlib.gridspec import GridSpec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

pd.set_option('display.max_columns', None)

params = {'font.family': 'Arial',
          'backend': 'ps',
          'axes.labelsize': 25,
          'grid.linewidth': 0.2,
          'font.size': 25,
          'legend.fontsize': 18,
          'legend.frameon': False,
          'xtick.labelsize': 30,
          'xtick.direction': 'out',
          'ytick.labelsize': 30,
          'ytick.direction': 'out',
          'legend.handlelength': 1,
          'legend.handleheight': 1,
          'savefig.bbox': 'tight',
          'axes.unicode_minus': False,
          "mathtext.default":"regular",
          'text.usetex': False}
rcParams.update(params)

def define_colormap(levels, cmap):
    """Define custom colormap and normalization."""
    # Generate colormap
    cmap = plt.get_cmap(cmap)
    color = cmap(np.linspace(0, 1, len(levels) - 1))
    cmap_custom = colors.ListedColormap(color)
    cmap_custom.set_under(cmap(0))
    cmap_custom.set_over(color[-1])
    norm = colors.BoundaryNorm(levels, cmap_custom.N)
    return cmap_custom, norm

def draw(path,name,level,cmap,region,*norm):
    ds = xr.open_dataset(path).sel(lon=slice(region[0], region[1]), lat=slice(region[3], region[2]))
    data = ds['f_annavg_somhr'][0,:,:]*365*86400
    print(data.min(),data.max())
    # data = np.ma.masked_where(np.isnan(data), data)
    # data = np.flipud(data)
    
    fig = plt.figure(figsize=(12, 6), dpi=300)

    fig.subplots_adjust(left=0.05, right=0.98, 
                    bottom=0.14, top=0.95, hspace=0.25) 
        
    #Create a subgraph grid with 2 rows and 3 columns
    gs = GridSpec(2, 6)
    ax = fig.add_subplot(gs[:, :], projection=ccrs.PlateCarree())

    # Set drawing mode(note:region's lat from positive to negative)
    img = ax.imshow(data, cmap=cmap, extent=region, vmin=level[0],vmax=level[-1])

    for spine in ax.spines.values():
        spine.set_edgecolor('black')  
        spine.set_linewidth(0)  

    ax.set_xlim(region[0], region[1])
    ax.set_ylim(region[2], region[3])

    # coastline = cfeature.NaturalEarthFeature('physical', 'coastline', '50m', edgecolor='0.6', facecolor='none')
    rivers = cfeature.NaturalEarthFeature('physical', 'rivers_lake_centerlines', '110m', edgecolor='0.6', facecolor='none')
    ax.add_feature(cfeature.LAND, facecolor='0.95')
    # ax.add_feature(coastline, linewidth=0.6)
    ax.add_feature(cfeature.LAKES, alpha=1, facecolor='white', edgecolor='white')
    ax.add_feature(rivers, linewidth=0.8)
    # ax.gridlines(draw_labels=False, linestyle=':', linewidth=0.7, color='grey', alpha=0.8)

    ax.add_feature(cfeature.COASTLINE)
    ax.set_extent(region)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())

    # ax.grid(ls = "--", lw = 0.25, color = "#4E616C")

    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=False, linestyle='--', color='#4E616C')   
    gl.xlocator = ticker.FixedLocator(np.arange(-120, 181, 60)) 
    gl.ylocator = ticker.FixedLocator(np.arange(-60, 91, 30))    

    # From the bottom left corner x, y, width, height
    custom_ticks = [0, 10, 20, 30, 40]
    # cbar_ax = fig.add_axes([0.4, 0.14, 0.4, 0.03], frameon = False) 
    cbar_ax = fig.add_axes([0.1, 0.06, 0.8, 0.04], frameon = True) 
    cb = fig.colorbar(img, 
                    drawedges=True,
                    ticks=level, 
                    cax=cbar_ax, 
                    orientation='horizontal',
                    spacing='uniform')

    cb.ax.tick_params(labelsize=20)
    cb.ax.yaxis.set_tick_params(direction='out', width=1.5)
    # for label in cb.ax.get_xticklabels() + cb.ax.get_yticklabels():
    #     label.set_fontproperties(font_properties)
    cb.set_label(f'{name[3]}', fontsize=30, fontweight='bold')

    cb.outline.set_visible(True)
    cb.outline.set_edgecolor('#333333')
    cb.outline.set_linewidth(2)

    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# 创建一个从灰到 RdYlGn_r 的组合 colormap
colors1 = plt.cm.Greys(np.linspace(0.3, 0.6, 64))   # 灰色部分
colors2 = plt.cm.RdYlGn_r(np.linspace(0, 1, 256))   # 原始绿黄红
colors = np.vstack((colors1, colors2))               # 拼接
cmap = mcolors.LinearSegmentedColormap.from_list('gray_RdYlGn_r', colors)

prefix='g1'
path = f'/share/home/dq076/data/cases/globe/{prefix}/postdata/annavg_somhr.nc'

region = [-180,180,-60,90]
name = [f'Sbedrock', 'Sbedrock', 'Sbedrock', r"Soil Heterotrophic Respiration (gC/$m^{2}$/yr)"]
level = [0, 300, 600, 900]
# cmap = cmaps.cmp_b2r
# rgb_list = ['#f6f6f5','#2f7c00','#64a201','#97b70d',
#                                 '#69aa4c','#CCCC00','#ebc874','#99004C','#FF6666']
# cmap = colors.ListedColormap(rgb_list)
# cmap = cmaps.
# cmap, norm = define_colormap(level, cmap)
draw(path,name,level,cmap,region)